In [34]:
import pandas as pd

In [35]:
from datasets import load_dataset

dataset=load_dataset("victor/real-or-fake-fake-jobposting-prediction")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent'],
        num_rows: 17880
    })
})


In [36]:
df=dataset["train"].to_pandas()

In [37]:
print(df.head())

   job_id                                      title            location  \
0       1                           Marketing Intern    US, NY, New York   
1       2  Customer Service - Cloud Video Production      NZ, , Auckland   
2       3    Commissioning Machinery Assistant (CMA)       US, IA, Wever   
3       4          Account Executive - Washington DC  US, DC, Washington   
4       5                        Bill Review Manager  US, FL, Fort Worth   

  department salary_range                                    company_profile  \
0  Marketing          NaN  We're Food52, and we've created a groundbreaki...   
1    Success          NaN  90 Seconds, the worlds Cloud Video Production ...   
2        NaN          NaN  Valor Services provides Workforce Solutions th...   
3      Sales          NaN  Our passion for improving quality of life thro...   
4        NaN          NaN  SpotSource Solutions LLC is a Global Human Cap...   

                                         description  \
0  Foo

In [38]:
print(df.shape)
print(df.columns)
print(df.info())

(17880, 18)
Index(['job_id', 'title', 'location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits',
       'telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   job_id               17880 non-null  int64
 1   title                17880 non-null  str  
 2   location             17534 non-null  str  
 3   department           6333 non-null   str  
 4   salary_range         2868 non-null   str  
 5   company_profile      14572 non-null  str  
 6   description          17879 non-null  str  
 7   requirements         15184 non-null  str  
 8   benefits             10668 non-null  str  
 9   telecommuting        17

In [39]:
print(df.columns.tolist())

print(df.isnull().sum().sort_values(ascending=False))

['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']
salary_range           15012
department             11547
required_education      8105
benefits                7212
required_experience     7050
function                6455
industry                4903
employment_type         3471
company_profile         3308
requirements            2696
location                 346
description                1
job_id                     0
telecommuting              0
has_questions              0
has_company_logo           0
title                      0
fraudulent                 0
dtype: int64


In [40]:
df=df.drop(columns=["job_id", "salary_range", "department"])

In [41]:
text_features = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

categorical_features = [
    "location",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

binary_features = [
    "telecommuting",
    "has_company_logo",
    "has_questions"
]

target = "fraudulent"

In [42]:
df[text_features] = df[text_features].fillna("")

df[categorical_features] = df[categorical_features].fillna("Unknown")

In [43]:
print(df.isnull().sum().sort_values(ascending=False))

title                  0
location               0
company_profile        0
description            0
requirements           0
benefits               0
telecommuting          0
has_company_logo       0
has_questions          0
employment_type        0
required_experience    0
required_education     0
industry               0
function               0
fraudulent             0
dtype: int64


In [44]:
X=df.drop("fraudulent",axis=1) #means use all features except our target which is fraudulent so we dropped it
y=df["fraudulent"]

In [45]:
from sklearn.model_selection import train_test_split

#stratify maintains the proper ratio of class distribution

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("Original:")
print(y.value_counts(normalize=True))

print("\nTraining:")
print(y_train.value_counts(normalize=True))

print("\nTesting:")
print(y_test.value_counts(normalize=True))


X_train: (14304, 14)
X_test: (3576, 14)
y_train: (14304,)
y_test: (3576,)
Original:
fraudulent
0    0.951566
1    0.048434
Name: proportion, dtype: float64

Training:
fraudulent
0    0.951552
1    0.048448
Name: proportion, dtype: float64

Testing:
fraudulent
0    0.951622
1    0.048378
Name: proportion, dtype: float64


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

categorical_features = [
    "location",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

binary_features = [
    "telecommuting",
    "has_company_logo",
    "has_questions"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("title", TfidfVectorizer(), "title"),
        ("company_profile", TfidfVectorizer(), "company_profile"),
        ("description", TfidfVectorizer(), "description"),
        ("requirements", TfidfVectorizer(), "requirements"),
        ("benefits", TfidfVectorizer(), "benefits"),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("binary", "passthrough", binary_features)
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression())
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.9840604026845637
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3403
           1       0.99      0.68      0.80       173

    accuracy                           0.98      3576
   macro avg       0.99      0.84      0.90      3576
weighted avg       0.98      0.98      0.98      3576

[[3402    1]
 [  56  117]]


In [50]:
from sklearn.svm import LinearSVC

svm_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearSVC())
])

svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.9904921700223713
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      3403
           1       0.98      0.82      0.89       173

    accuracy                           0.99      3576
   macro avg       0.99      0.91      0.94      3576
weighted avg       0.99      0.99      0.99      3576

[[3402    1]
 [  56  117]]


In [51]:
from sklearn.naive_bayes import MultinomialNB

nb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", MultinomialNB())
])

nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.9516219239373602
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      3403
           1       0.00      0.00      0.00       173

    accuracy                           0.95      3576
   macro avg       0.48      0.50      0.49      3576
weighted avg       0.91      0.95      0.93      3576

[[3402    1]
 [  56  117]]


/Users/tejasmac/ML Projects/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/tejasmac/ML Projects/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/tejasmac/ML Projects/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

In [52]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

knn_model.fit(X_train, y_train)

y_pred_knn = knn_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.9784675615212528
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3403
           1       0.81      0.72      0.76       173

    accuracy                           0.98      3576
   macro avg       0.90      0.86      0.88      3576
weighted avg       0.98      0.98      0.98      3576

[[3402    1]
 [  56  117]]


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=7,
        random_state=42
    ))
])

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__max_depth": [3, 5, 7, 9, 11, 13, 15, None]
}

grid_search = GridSearchCV(
    dt_model,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)



Accuracy: 0.9818232662192393
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3403
           1       0.94      0.67      0.78       173

    accuracy                           0.98      3576
   macro avg       0.96      0.83      0.89      3576
weighted avg       0.98      0.98      0.98      3576

[[3402    1]
 [  56  117]]
Best Parameters: {'model__max_depth': None}
Best CV F1: 0.7808880895004597


In [57]:
best_dt = grid_search.best_estimator_

y_pred_best_dt = best_dt.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_best_dt))
print(classification_report(y_test, y_pred_best_dt))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.9837807606263982
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3403
           1       0.82      0.84      0.83       173

    accuracy                           0.98      3576
   macro avg       0.91      0.92      0.91      3576
weighted avg       0.98      0.98      0.98      3576

[[3402    1]
 [  56  117]]


In [59]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

from sklearn.model_selection import GridSearchCV

param_grid_rf = {
    "model__n_estimators": [50, 100, 150],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5]
}

grid_search_rf = GridSearchCV(
    rf_model,
    param_grid_rf,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

print("Best Parameters:", grid_search_rf.best_params_)
print("Best CV F1:", grid_search_rf.best_score_)

Accuracy: 0.9837807606263982

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3403
           1       1.00      0.66      0.80       173

    accuracy                           0.98      3576
   macro avg       0.99      0.83      0.90      3576
weighted avg       0.98      0.98      0.98      3576


Confusion Matrix:
[[3403    0]
 [  58  115]]
Best Parameters: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Best CV F1: 0.7688606204194768


In [62]:
import joblib

joblib.dump(svm_model, "../src/jobshield_model.pkl")

print("Final model saved successfully!")

Final model saved successfully!


In [63]:
svm_results = {
    "Model": ["Linear SVM"],
    "Accuracy": [0.9905],
    "Fake Precision": [0.98],
    "Fake Recall": [0.82],
    "Fake F1": [0.89]
}

results_df = pd.DataFrame(svm_results)

results_df.to_csv("../results/svm_results.csv", index=False)

print("SVM results saved successfully!")

SVM results saved successfully!
